# Training a YOLO11 model on Google Colab using a T4

In [2]:
!uv tree

Resolved 85 packages in 6ms
submission v0.1.0
├── numpy v1.26.4
├── torch v2.6.0
│   ├── filelock v3.25.2
│   ├── fsspec v2026.2.0
│   ├── jinja2 v3.1.6
│   │   └── markupsafe v3.0.3
│   ├── networkx v3.6.1
│   ├── setuptools v82.0.1
│   ├── sympy v1.13.1
│   │   └── mpmath v1.3.0
│   └── typing-extensions v4.15.0
├── torchvision v0.21.0
│   ├── numpy v1.26.4
│   ├── pillow v12.1.1
│   └── torch v2.6.0 (*)
├── ultralytics v8.1.0
│   ├── hub-sdk v0.0.24
│   │   └── requests v2.32.5
│   │       ├── certifi v2026.2.25
│   │       ├── charset-normalizer v3.4.6
│   │       ├── idna v3.11
│   │       └── urllib3 v2.6.3
│   ├── matplotlib v3.10.8
│   │   ├── contourpy v1.3.3
│   │   │   └── numpy v1.26.4
│   │   ├── cycler v0.12.1
│   │   ├── fonttools v4.62.1
│   │   ├── kiwisolver v1.5.0
│   │   ├── numpy v1.26.4
│   │   ├── packaging v26.0
│   │   ├── pillow v12.1.1
│   │   ├── pyparsing v3.3.2
│   │   └── python-dateutil v2.9.0.post0
│   │       └── six v1.17.0
│   ├── numpy v1.26.4
│   ├

In [3]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


## Upload data
Set up the data in the correct format and upload a data.zip file

In [ ]:
!unzip -q /content/data.zip -d /content/custom_data 

## Split images into train and validation folders
a Python script that will automatically create the required folder structure and randomly move 90% of dataset to the "train" folder and 10% to the "validation" folder. Run the following code block to download and execute the scrpt.

In [ ]:
!wget -O /content/train_val_split.py https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py

# TO DO: Improve robustness of train_val_split.py script so it can handle nested data folders, etc
!python train_val_split.py --datapath="./custom_data" --train_pct=0.9

## Configure training

In [ ]:
# Python function to automatically create data.yaml config file
# 1. Reads "classes.txt" file to get list of class names
# 2. Creates data dictionary with correct paths to folders, number of classes, and names of classes
# 3. Writes data in YAML format to data.yaml

import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  # Read class.txt to get class names
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  # Create data dictionary
  data = {
      'path': '/content/data',
      'train': 'train/images',
      'val': 'validation/images',
      'nc': number_of_classes,
      'names': classes
  }

  # Write data to YAML file
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Define path to classes.txt and run function
path_to_classes_txt = '/content/custom_data/classes.txt'
path_to_data_yaml = '/content/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

print('\nFile contents:\n')
!cat /content/data.yaml

## Train model

### Training Parameters
Now that the data is organized and the config file is created, we're ready to start training! First, there are a few important parameters to decide on. Visit my article on [Training YOLO Models Locally](https://www.ejtech.io/learn/train-yolo-models) to learn more about these parameters and how to choose them.

**Model architecture & size (`model`):**

There are several YOLO11 models sizes available to train, including `yolo11n.pt`, `yolo11s.pt`, `yolo11m.pt`, `yolo11l.pt`, and `yolo11xl.pt`. Larger models run slower but have higher accuracy, while smaller models run faster but have lower accuracy. I made a brief YouTube video that compares performance of different YOLO models on a Raspberry Pi 5 and a laptop with a RTX 4050 GPU, [check it out here to get a sense of their speed accuracy](https://youtu.be/_WKS4E9SmkA). If you aren't sure which model size to use, `yolo11s.pt` is a good starting point.

You can also train YOLOv8 or YOLOv5 models by substituting `yolo11` for `yolov8` or `yolov5`.


**Number of epochs (`epochs`)**

In machine learning, one “epoch” is one single pass through the full training dataset. Setting the number of epochs dictates how long the model will train for. The best amount of epochs to use depends on the size of the dataset and the model architecture. If your dataset has less than 200 images, a good starting point is 60 epochs. If your dataset has more than 200 images, a good starting point is 40 epochs.


**Resolution (`imgsz`)**

Resolution has a large impact on the speed and accuracy of the model: a lower resolution model will have higher speed but less accuracy. YOLO models are typically trained and inferenced at a 640x640 resolution. However, if you want your model to run faster or know you will be working with low-resolution images, try using a lower resolution like 480x480.


### Run Training
Run the following code block to begin training. If you want to use a different model, number of epochs, or resolution, change model, epochs, or imgsz.

In [ ]:
!yolo detect train data=/content/data.yaml model=yolo11s.pt epochs=200 imgsz=640

## Test the best model
The model has been trained; now it's time to test it! The commands below run the model on the images in the validation folder and then display the results for the first 10 images. This is a good way to confirm your model is working as expected. Click Play on the blocks below to see how your model performs.

In [ ]:
!yolo detect predict model=runs/detect/train2/weights/best.pt source=data/validation/images save=True

The model should draw a box around each object of interest in each image. If it isn't doing a good job of detecting objects, here are a few tips:

1. Double-check your dataset to make sure there are no labeling errors or conflicting examples.
2. Increase the number of epochs used for training.
Use a larger model size (e.g. yolo11l.pt).
3. Add more images to the training dataset. 

You can also run the model on video files or other images images by uploading them to this notebook and using the above !yolo detect predict command, where source points to the location of the video file, image, or folder of images. The results will be saved in runs/detect/predict.

Drawing boxes on images is great, but it isn't very useful in itself. It's also not very helpful to just run this models inside a Colab notebook: it's easier if we can just run it on a local computer. Continue to the next section to see how to download your newly trained model and run it on a local device.



In [ ]:
import glob
from IPython.display import Image, display
for image_path in glob.glob(f'/content/runs/detect/predict/*.jpg')[:10]:
  display(Image(filename=image_path, height=400))
  print('\n')


In [ ]:
# Create "my_model" folder to store model weights and train results
!mkdir /content/my_model
!cp /content/runs/detect/train2/weights/best.pt /content/my_model/my_model.pt
!cp -r /content/runs/detect/train2 /content/my_model

In [ ]:
# Zip into "my_model.zip"
%cd my_model
!zip /content/my_model.zip my_model.pt
!zip -r /content/my_model.zip train2
%cd /content